# Time Series Features

When working with many time series at once, it is useful to compute **summary
features** that describe each series in a small number of numbers.  These
features can be used for clustering, classification, anomaly detection, or
simply for understanding a large collection of series.

This notebook covers:
1. Summary statistics (mean, variance, ACF at lag 1)
2. Trend strength $F_T$ and seasonal strength $F_S$ from STL
3. Spectral entropy (brief)
4. Feature extraction across multiple datasets
5. Comparing and visualising features

## Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import acf
from statsmodels.tsa.seasonal import STL

# Plotting defaults
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 100
plt.rcParams["lines.linewidth"] = 1.5

DATA_DIR = Path("../data")

In [ ]:
# Load several monthly datasets for comparison
datasets = {}

# Airline Passengers
df = pd.read_csv(DATA_DIR / "airline_passengers.csv", index_col="Month", parse_dates=True)
df.index.freq = "MS"
datasets["Airline"] = df.iloc[:, 0]

# Alcohol Sales
df = pd.read_csv(DATA_DIR / "Alcohol_Sales.csv", index_col="DATE", parse_dates=True)
df.index.freq = "MS"
datasets["Alcohol"] = df.iloc[:, 0]

# Energy Production
df = pd.read_csv(DATA_DIR / "EnergyProduction.csv", index_col="DATE", parse_dates=True)
df.index.freq = "MS"
datasets["Energy"] = df.iloc[:, 0]

# Monthly Milk Production
df = pd.read_csv(DATA_DIR / "monthly_milk_production.csv", index_col="Date", parse_dates=True)
df.index.freq = "MS"
datasets["Milk"] = df.iloc[:, 0]

# Hospitality Employees
df = pd.read_csv(DATA_DIR / "HospitalityEmployees.csv", index_col="DATE", parse_dates=True)
df.index.freq = "MS"
datasets["Hospitality"] = df.iloc[:, 0]

# Miles Traveled
df = pd.read_csv(DATA_DIR / "Miles_Traveled.csv", index_col="DATE", parse_dates=True)
df.index.freq = "MS"
datasets["Miles"] = df.iloc[:, 0]

print("Loaded datasets:")
for name, s in datasets.items():
    print(f"  {name:<15} {s.shape[0]:>5} obs  ({s.index[0].date()} to {s.index[-1].date()})")

---
## 1. Basic Summary Statistics

The simplest features: mean, standard deviation, coefficient of variation,
and autocorrelation at lag 1.

In [ ]:
basic_features = []

for name, series in datasets.items():
    s = series.dropna()
    acf_1 = acf(s, nlags=1, fft=True)[1]
    basic_features.append({
        "Series": name,
        "N": len(s),
        "Mean": s.mean(),
        "Std": s.std(),
        "CV": s.std() / s.mean(),
        "Min": s.min(),
        "Max": s.max(),
        "ACF(1)": acf_1,
    })

basic_df = pd.DataFrame(basic_features)
print(basic_df.to_string(index=False, float_format="{:.4f}".format))

In [ ]:
# Visualise ACF(1) across datasets
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(basic_df["Series"], basic_df["ACF(1)"], color="steelblue", alpha=0.8)
ax.set_xlabel("ACF at Lag 1")
ax.set_title("Autocorrelation at Lag 1 — All Datasets")
ax.axvline(0, color="black", linewidth=0.5)
plt.tight_layout()
plt.show()

print("All series show high positive ACF(1), which is typical for trended monthly data.")

---
## 2. Trend Strength and Seasonal Strength

These features, proposed by Wang, Smith, and Hyndman (2006) and used in
FPP3 Chapter 4, quantify how much of the variation is due to trend and
seasonality.  They are computed from an **STL decomposition**.

### Trend Strength

$$
F_T = \max\left(0, \; 1 - \frac{\text{Var}(R_t)}{\text{Var}(T_t + R_t)}\right)
$$

where $T_t$ is the trend component and $R_t$ is the remainder.

- $F_T \approx 0$: no trend (remainder dominates)
- $F_T \approx 1$: strong trend (remainder is small relative to trend)

### Seasonal Strength

$$
F_S = \max\left(0, \; 1 - \frac{\text{Var}(R_t)}{\text{Var}(S_t + R_t)}\right)
$$

where $S_t$ is the seasonal component.

- $F_S \approx 0$: no seasonality
- $F_S \approx 1$: strong seasonality

In [ ]:
def compute_strength(series, period=12):
    """Compute trend and seasonal strength from STL decomposition.

    Parameters
    ----------
    series : pd.Series
        The time series (must have a frequency set).
    period : int
        Seasonal period (default 12 for monthly data).

    Returns
    -------
    dict with keys 'F_trend' and 'F_seasonal'.
    """
    stl = STL(series.dropna(), period=period, robust=True)
    result = stl.fit()

    trend = result.trend
    seasonal = result.seasonal
    remainder = result.resid

    # Trend strength
    var_remainder = np.var(remainder)
    var_trend_plus_remainder = np.var(trend + remainder)
    f_trend = max(0, 1 - var_remainder / var_trend_plus_remainder)

    # Seasonal strength
    var_seasonal_plus_remainder = np.var(seasonal + remainder)
    f_seasonal = max(0, 1 - var_remainder / var_seasonal_plus_remainder)

    return {"F_trend": f_trend, "F_seasonal": f_seasonal}

In [ ]:
# Demonstrate with airline passengers
airline_strength = compute_strength(datasets["Airline"])

print("Airline Passengers:")
print(f"  Trend strength:    F_T = {airline_strength['F_trend']:.4f}")
print(f"  Seasonal strength: F_S = {airline_strength['F_seasonal']:.4f}")
print()
print("Both are high — the series has a strong trend AND strong seasonality.")

In [ ]:
# Visualise the STL components for airline to see why strengths are high
stl_airline = STL(datasets["Airline"].dropna(), period=12, robust=True).fit()

fig = stl_airline.plot()
fig.set_size_inches(12, 10)
fig.suptitle("STL Decomposition — Airline Passengers", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Compute strengths for all datasets
strength_features = []

for name, series in datasets.items():
    strengths = compute_strength(series)
    strength_features.append({
        "Series": name,
        "F_trend": strengths["F_trend"],
        "F_seasonal": strengths["F_seasonal"],
    })

strength_df = pd.DataFrame(strength_features)
print(strength_df.to_string(index=False, float_format="{:.4f}".format))

In [ ]:
# Scatter plot: trend strength vs seasonal strength
fig, ax = plt.subplots(figsize=(8, 6))

ax.scatter(strength_df["F_trend"], strength_df["F_seasonal"],
           s=100, edgecolors="black", linewidths=0.5, zorder=3)

for _, row in strength_df.iterrows():
    ax.annotate(row["Series"],
                (row["F_trend"], row["F_seasonal"]),
                textcoords="offset points", xytext=(8, 5),
                fontsize=10)

ax.set_xlabel("Trend Strength $F_T$", fontsize=12)
ax.set_ylabel("Seasonal Strength $F_S$", fontsize=12)
ax.set_title("Trend vs Seasonal Strength — All Datasets", fontsize=14)
ax.set_xlim(-0.05, 1.05)
ax.set_ylim(-0.05, 1.05)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("This plot immediately reveals which series are driven by trend,")
print("by seasonality, or by both — useful for selecting appropriate models.")

---
## 3. Spectral Entropy

**Spectral entropy** measures the "complexity" or "unpredictability" of a
time series based on its power spectrum.  It is normalised to $[0, 1]$:

- $H \approx 0$: highly predictable (energy concentrated at a few frequencies)
- $H \approx 1$: maximally unpredictable (flat spectrum, like white noise)

The computation:
1. Compute the power spectral density (PSD) using the periodogram
2. Normalise to get a probability distribution $p_i$
3. Compute the Shannon entropy: $H = -\sum p_i \log_2 p_i$
4. Normalise by $\log_2(N)$ to get $H \in [0, 1]$

In [ ]:
def spectral_entropy(series):
    """Compute the normalised spectral entropy of a time series.

    Parameters
    ----------
    series : array-like
        The time series values.

    Returns
    -------
    float : Spectral entropy in [0, 1].
    """
    x = np.asarray(series, dtype=float)
    x = x - x.mean()  # centre

    # Power spectral density via FFT
    fft_vals = np.fft.rfft(x)
    psd = np.abs(fft_vals) ** 2

    # Normalise to probability distribution
    psd_norm = psd / psd.sum()
    psd_norm = psd_norm[psd_norm > 0]  # avoid log(0)

    # Shannon entropy, normalised
    entropy = -np.sum(psd_norm * np.log2(psd_norm))
    max_entropy = np.log2(len(psd))

    return entropy / max_entropy

In [ ]:
# Compute spectral entropy for all datasets
print("Spectral Entropy:")
print("="*35)
entropies = {}
for name, series in datasets.items():
    se = spectral_entropy(series.dropna())
    entropies[name] = se
    print(f"  {name:<15} H = {se:.4f}")

# White noise reference
np.random.seed(42)
wn_entropy = spectral_entropy(np.random.randn(500))
print(f"  {'White noise':<15} H = {wn_entropy:.4f}")
print()
print("Lower entropy => more predictable (energy concentrated at few frequencies).")
print("Higher entropy => more like white noise (flat spectrum).")

In [ ]:
# Bar chart of spectral entropy
ent_df = pd.Series(entropies)
ent_df["White Noise"] = wn_entropy

fig, ax = plt.subplots(figsize=(10, 5))
colors = ["steelblue"] * len(datasets) + ["tab:red"]
ent_df.plot.barh(ax=ax, color=colors, alpha=0.8)
ax.set_xlabel("Spectral Entropy")
ax.set_title("Spectral Entropy — Predictability of Each Series")
ax.set_xlim(0, 1)
plt.tight_layout()
plt.show()

---
## 4. Comprehensive Feature Extraction

Let's combine all the features into a single table.

In [ ]:
def extract_features(series, name, period=12):
    """Extract a comprehensive set of features from a time series.

    Parameters
    ----------
    series : pd.Series
        The time series.
    name : str
        Label for the series.
    period : int
        Seasonal period for STL.

    Returns
    -------
    dict of features.
    """
    s = series.dropna()
    n = len(s)
    acf_vals = acf(s, nlags=min(period, n // 2), fft=True)

    # Trend and seasonal strength
    strengths = compute_strength(s, period=period)

    # Spectral entropy
    se = spectral_entropy(s)

    return {
        "Series": name,
        "N": n,
        "Mean": s.mean(),
        "Std": s.std(),
        "CV": s.std() / abs(s.mean()) if s.mean() != 0 else np.nan,
        "ACF(1)": acf_vals[1],
        "ACF(12)": acf_vals[period] if len(acf_vals) > period else np.nan,
        "F_trend": strengths["F_trend"],
        "F_seasonal": strengths["F_seasonal"],
        "Spectral Entropy": se,
    }

In [ ]:
all_features = []
for name, series in datasets.items():
    features = extract_features(series, name)
    all_features.append(features)

features_df = pd.DataFrame(all_features)
print(features_df.to_string(index=False, float_format="{:.4f}".format))

In [ ]:
# Heatmap of normalised features (min-max scaling for visual comparison)
numeric_cols = ["CV", "ACF(1)", "ACF(12)", "F_trend", "F_seasonal", "Spectral Entropy"]
norm_df = features_df[numeric_cols].copy()

# Min-max normalise each column
for col in numeric_cols:
    col_min = norm_df[col].min()
    col_max = norm_df[col].max()
    if col_max > col_min:
        norm_df[col] = (norm_df[col] - col_min) / (col_max - col_min)
    else:
        norm_df[col] = 0.5

fig, ax = plt.subplots(figsize=(10, 5))
im = ax.imshow(norm_df.values, cmap="YlOrRd", aspect="auto")

ax.set_xticks(range(len(numeric_cols)))
ax.set_xticklabels(numeric_cols, rotation=45, ha="right")
ax.set_yticks(range(len(features_df)))
ax.set_yticklabels(features_df["Series"])
ax.set_title("Feature Heatmap (min-max normalised)", fontsize=14)

# Annotate cells
for i in range(norm_df.shape[0]):
    for j in range(norm_df.shape[1]):
        val = features_df[numeric_cols[j]].iloc[i]
        ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=9)

plt.colorbar(im, label="Normalised Value")
plt.tight_layout()
plt.show()

---
## 5. Comparing Datasets by Features

In [ ]:
# Multi-panel comparison: all datasets on one figure
n_datasets = len(datasets)
fig, axes = plt.subplots(n_datasets, 1, figsize=(14, 3 * n_datasets), sharex=False)

for ax, (name, series) in zip(axes, datasets.items()):
    s = series.dropna()
    ax.plot(s)
    ax.set_title(name, fontsize=11, loc="left")
    ax.set_ylabel("Value")

plt.suptitle("All Monthly Datasets", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Pair plot of key features: F_trend vs F_seasonal vs Spectral Entropy
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# F_trend vs F_seasonal
axes[0].scatter(features_df["F_trend"], features_df["F_seasonal"],
                s=100, edgecolors="black", linewidths=0.5)
for _, row in features_df.iterrows():
    axes[0].annotate(row["Series"], (row["F_trend"], row["F_seasonal"]),
                     textcoords="offset points", xytext=(8, 5), fontsize=9)
axes[0].set_xlabel("Trend Strength $F_T$")
axes[0].set_ylabel("Seasonal Strength $F_S$")
axes[0].set_title("Trend vs Seasonal Strength")

# F_trend vs Spectral Entropy
axes[1].scatter(features_df["F_trend"], features_df["Spectral Entropy"],
                s=100, edgecolors="black", linewidths=0.5)
for _, row in features_df.iterrows():
    axes[1].annotate(row["Series"], (row["F_trend"], row["Spectral Entropy"]),
                     textcoords="offset points", xytext=(8, 5), fontsize=9)
axes[1].set_xlabel("Trend Strength $F_T$")
axes[1].set_ylabel("Spectral Entropy")
axes[1].set_title("Trend vs Entropy")

# F_seasonal vs Spectral Entropy
axes[2].scatter(features_df["F_seasonal"], features_df["Spectral Entropy"],
                s=100, edgecolors="black", linewidths=0.5)
for _, row in features_df.iterrows():
    axes[2].annotate(row["Series"], (row["F_seasonal"], row["Spectral Entropy"]),
                     textcoords="offset points", xytext=(8, 5), fontsize=9)
axes[2].set_xlabel("Seasonal Strength $F_S$")
axes[2].set_ylabel("Spectral Entropy")
axes[2].set_title("Seasonality vs Entropy")

for ax in axes:
    ax.grid(True, alpha=0.3)

plt.suptitle("Feature Space Projections", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("These plots reveal natural groupings among the datasets:")
print("  - Strong-trend, strong-seasonal: Airline, Alcohol, Miles")
print("  - Lower entropy => more structured/predictable series")

In [ ]:
# Rank datasets by each feature
print("Rankings (highest = 1):")
print("="*70)
for col in ["F_trend", "F_seasonal", "Spectral Entropy", "ACF(1)"]:
    ascending = col == "Spectral Entropy"  # lower entropy = more predictable
    ranked = features_df.sort_values(col, ascending=ascending)["Series"].tolist()
    print(f"  {col:<20}: {' > '.join(ranked)}")

---
## 6. Using Features for Model Selection Hints

Features can provide quick guidance on what kind of model might be
appropriate:

| Feature Pattern | Suggested Action |
|-----------------|------------------|
| High $F_T$, low $F_S$ | Differencing for trend; no seasonal component |
| High $F_S$, low $F_T$ | Seasonal model needed; may not need trend differencing |
| Both high | SARIMA or seasonal ETS with trend |
| High spectral entropy | Series may be hard to forecast (close to white noise) |
| High ACF(1) | Strong persistence — autoregressive terms likely needed |

In [ ]:
# Quick model suggestions based on features
print("Quick Model Suggestions Based on Features:")
print("=" * 60)
for _, row in features_df.iterrows():
    suggestions = []
    if row["F_trend"] > 0.5:
        suggestions.append("trend differencing (d=1)")
    if row["F_seasonal"] > 0.5:
        suggestions.append("seasonal differencing (D=1)")
    if row["ACF(1)"] > 0.8:
        suggestions.append("AR term(s) likely")
    if row["Spectral Entropy"] > 0.9:
        suggestions.append("may be hard to forecast")

    print(f"  {row['Series']:<15}: {'; '.join(suggestions) if suggestions else 'relatively simple'}")

---
## Summary

| Feature | Formula / Source | What It Tells You |
|---------|-----------------|-------------------|
| Mean, Std, CV | Basic statistics | Scale and variability |
| ACF(1) | Autocorrelation at lag 1 | Persistence / smoothness |
| ACF($m$) | Autocorrelation at seasonal lag | Strength of seasonal pattern |
| $F_T$ | $1 - \text{Var}(R) / \text{Var}(T+R)$ from STL | Trend strength |
| $F_S$ | $1 - \text{Var}(R) / \text{Var}(S+R)$ from STL | Seasonal strength |
| Spectral entropy | Shannon entropy of normalised PSD | Predictability (0=predictable, 1=noise) |

**Use cases:**
- **Comparing** many series at a glance
- **Clustering** similar series together
- **Guiding model selection** before fitting
- **Anomaly detection** — a series whose features differ markedly may need special treatment

In [ ]:
print("Chapter 05 complete.")
print()
print("We now have the diagnostic tools needed for model building:")
print("  - ACF/PACF for identifying AR/MA orders (Notebook 01)")
print("  - ADF/KPSS for testing stationarity (Notebook 02)")
print("  - Ljung-Box for checking residuals, Granger for causality (Notebook 03)")
print("  - Feature extraction for comparing and selecting models (Notebook 04)")
print()
print("Next: Chapter 06 — ARIMA Models")